# Disguise-Resistant Face & Person IdentifierFull pipeline: dataset setup, re-ID training (baseline + batch-hard mining), face detector, gallery matching, fusion pipeline.

## 1. Data setup

In [ ]:
import kagglehubpath = kagglehub.dataset_download("canomercik/wider-face-dataset-for-yolov12-format")print("Path to dataset files:", path)

In [ ]:
import kagglehubpath = kagglehub.dataset_download("pengcw1/market-1501")print("Path to dataset files:", path)

In [ ]:
import osdef preview_tree(path, max_depth=2, max_items=5):    for root, dirs, files in os.walk(path):        depth = root[len(path):].count(os.sep)        if depth > max_depth:            dirs[:] = []            continue        indent = "  " * depth        print(f"{indent}{os.path.basename(root)}/")        for f in files[:max_items]:            print(f"{indent}  {f}")        if len(files) > max_items:            print(f"{indent}  ... ({len(files)} files total)")wider_root = "/root/.cache/kagglehub/datasets/canomercik/wider-face-dataset-for-yolov12-format/versions/5"market_path = "/root/.cache/kagglehub/datasets/pengcw1/market-1501/versions/1"print("=== WIDER FACE ===")preview_tree(wider_root)print("\n=== Market-1501 ===")preview_tree(market_path)

## 2. Mount Google Drive (do this early so checkpoint saves never fail)

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

## 3. Market-1501 dataset class (triplet sampler)

In [ ]:
import osimport refrom collections import defaultdictfrom PIL import Imageimport randomclass Market1501Dataset:    """    Parses Market-1501 filenames: {person_id}_{camera_seq}_{frame}_{box_idx}.jpg    e.g. 0243_c1s1_066731_01.jpg -> person_id=243, camera=1    Junk images (person_id == -1) are distractors/background, excluded.    """    def __init__(self, root_dir, split="bounding_box_train"):        self.dir = os.path.join(root_dir, "Market-1501-v15.09.15", split)        self.pattern = re.compile(r'(-?\d+)_c(\d)s(\d)_(\d+)_(\d+)')        self.samples = []        self.identity_to_indices = defaultdict(list)        for fname in os.listdir(self.dir):            if not fname.endswith(".jpg"):                continue            m = self.pattern.match(fname)            if not m:                continue            person_id = int(m.group(1))            camera_id = int(m.group(2))            if person_id == -1:                continue            idx = len(self.samples)            self.samples.append((os.path.join(self.dir, fname), person_id, camera_id))            self.identity_to_indices[person_id].append(idx)        self.identities = list(self.identity_to_indices.keys())        print(f"Loaded {len(self.samples)} images, {len(self.identities)} identities from {split}")    def __len__(self):        return len(self.samples)    def get_triplet(self):        anchor_id = random.choice(self.identities)        pos_indices = self.identity_to_indices[anchor_id]        while len(pos_indices) < 2:            anchor_id = random.choice(self.identities)            pos_indices = self.identity_to_indices[anchor_id]        anchor_idx, pos_idx = random.sample(pos_indices, 2)        neg_id = random.choice(self.identities)        while neg_id == anchor_id:            neg_id = random.choice(self.identities)        neg_idx = random.choice(self.identity_to_indices[neg_id])        return self.samples[anchor_idx], self.samples[pos_idx], self.samples[neg_idx]    def load_image(self, path, size=(128, 64)):        img = Image.open(path).convert("RGB")        return img.resize(size[::-1])train_ds = Market1501Dataset(market_path, split="bounding_box_train")a, p, n = train_ds.get_triplet()print("Anchor:", a[1], "| Positive:", p[1], "| Negative:", n[1])

## 4. Re-ID embedding model (TensorFlow, ResNet50 backbone)

In [ ]:
import tensorflow as tffrom tensorflow.keras import layers, Modelfrom tensorflow.keras.applications import ResNet50from tensorflow.keras.applications.resnet50 import preprocess_inputimport numpy as npdef build_reid_embedding_model(embedding_dim=256, input_shape=(256, 128, 3)):    base = ResNet50(weights="imagenet", include_top=False, pooling="avg", input_shape=input_shape)    x = base.output    x = layers.Dense(embedding_dim)(x)    x = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)    return Model(inputs=base.input, outputs=x, name="reid_embedding_net")model = build_reid_embedding_model(embedding_dim=256)model.summary()

## 5. Stage 1 training: random triplet sampling (baseline)

In [ ]:
def triplet_loss(margin=0.3):    def loss_fn(anchor, positive, negative):        pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=1)        neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=1)        basic_loss = pos_dist - neg_dist + margin        return tf.reduce_mean(tf.maximum(basic_loss, 0.0))    return loss_fnloss_fn = triplet_loss(margin=0.3)optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)IMG_SIZE = (256, 128)def load_and_preprocess(path):    img = tf.io.read_file(path)    img = tf.image.decode_jpeg(img, channels=3)    img = tf.image.resize(img, IMG_SIZE)    img = preprocess_input(img)    return imgdef triplet_generator(market_dataset, num_triplets=5000):    def gen():        for _ in range(num_triplets):            a, p, n = market_dataset.get_triplet()            yield a[0], p[0], n[0]    return gendef make_triplet_dataset(market_dataset, num_triplets=5000, batch_size=32):    ds = tf.data.Dataset.from_generator(        triplet_generator(market_dataset, num_triplets),        output_signature=(            tf.TensorSpec(shape=(), dtype=tf.string),            tf.TensorSpec(shape=(), dtype=tf.string),            tf.TensorSpec(shape=(), dtype=tf.string),        )    )    ds = ds.map(        lambda a, p, n: (load_and_preprocess(a), load_and_preprocess(p), load_and_preprocess(n)),        num_parallel_calls=tf.data.AUTOTUNE    )    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)    return dstriplet_ds = make_triplet_dataset(train_ds, num_triplets=5000, batch_size=32)

In [ ]:
EPOCHS = 15@tf.functiondef train_step(a_batch, p_batch, n_batch):    with tf.GradientTape() as tape:        emb_a = model(a_batch, training=True)        emb_p = model(p_batch, training=True)        emb_n = model(n_batch, training=True)        loss = loss_fn(emb_a, emb_p, emb_n)    grads = tape.gradient(loss, model.trainable_variables)    optimizer.apply_gradients(zip(grads, model.trainable_variables))    return lossfor epoch in range(EPOCHS):    epoch_loss = tf.keras.metrics.Mean()    for a_batch, p_batch, n_batch in triplet_ds:        loss = train_step(a_batch, p_batch, n_batch)        epoch_loss.update_state(loss)    print(f"Epoch {epoch+1}/{EPOCHS} - triplet loss: {epoch_loss.result():.4f}")model.save("/content/reid_embedding_resnet50.keras")import shutilshutil.copy("/content/reid_embedding_resnet50.keras", "/content/drive/MyDrive/reid_embedding_resnet50.keras")

## 6. Stage 2 training: batch-hard triplet mining (the upgrade that got Rank-1 from 39% to 74%)

In [ ]:
def pk_batch_generator(market_dataset, P=16, K=4, num_batches=300):    """P identities per batch, K images per identity -> batch size P*K."""    eligible_ids = [pid for pid, idxs in market_dataset.identity_to_indices.items() if len(idxs) >= K]    def gen():        for _ in range(num_batches):            batch_ids = random.sample(eligible_ids, P)            batch_paths = []            batch_labels = []            for pid in batch_ids:                idxs = random.sample(market_dataset.identity_to_indices[pid], K)                for idx in idxs:                    path, person_id, cam_id = market_dataset.samples[idx]                    batch_paths.append(path)                    batch_labels.append(person_id)            yield batch_paths, batch_labels    return gendef make_pk_dataset(market_dataset, P=16, K=4, num_batches=300, img_size=(256, 128)):    def load_and_preprocess_local(path):        img = tf.io.read_file(path)        img = tf.image.decode_jpeg(img, channels=3)        img = tf.image.resize(img, img_size)        img = preprocess_input(img)        return img    gen = pk_batch_generator(market_dataset, P, K, num_batches)    def wrapped_gen():        for paths, labels in gen():            imgs = tf.stack([load_and_preprocess_local(p) for p in paths])            yield imgs, tf.constant(labels, dtype=tf.int32)    ds = tf.data.Dataset.from_generator(        wrapped_gen,        output_signature=(            tf.TensorSpec(shape=(P*K, *img_size, 3), dtype=tf.float32),            tf.TensorSpec(shape=(P*K,), dtype=tf.int32),        )    )    return ds.prefetch(tf.data.AUTOTUNE)def batch_hard_triplet_loss(embeddings, labels, margin=0.3):    dot_product = tf.matmul(embeddings, embeddings, transpose_b=True)    square_norm = tf.linalg.diag_part(dot_product)    dist_matrix = tf.expand_dims(square_norm, 1) - 2.0 * dot_product + tf.expand_dims(square_norm, 0)    dist_matrix = tf.maximum(dist_matrix, 0.0)    labels = tf.reshape(labels, [-1, 1])    same_identity_mask = tf.equal(labels, tf.transpose(labels))    diff_identity_mask = tf.logical_not(same_identity_mask)    identity_mask_no_self = tf.logical_and(same_identity_mask, tf.logical_not(tf.eye(tf.shape(labels)[0], dtype=tf.bool)))    pos_dist = tf.where(identity_mask_no_self, dist_matrix, -1.0)    hardest_positive = tf.reduce_max(pos_dist, axis=1)    max_dist = tf.reduce_max(dist_matrix)    neg_dist = tf.where(diff_identity_mask, dist_matrix, max_dist + 1.0)    hardest_negative = tf.reduce_min(neg_dist, axis=1)    loss = tf.maximum(hardest_positive - hardest_negative + margin, 0.0)    return tf.reduce_mean(loss)optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)pk_ds = make_pk_dataset(train_ds, P=16, K=4, num_batches=300)

In [ ]:
EPOCHS = 15@tf.functiondef train_step_hard(imgs, labels):    with tf.GradientTape() as tape:        embeddings = model(imgs, training=True)        loss = batch_hard_triplet_loss(embeddings, labels, margin=0.3)    grads = tape.gradient(loss, model.trainable_variables)    optimizer.apply_gradients(zip(grads, model.trainable_variables))    return lossfor epoch in range(EPOCHS):    epoch_loss = tf.keras.metrics.Mean()    for imgs, labels in pk_ds:        loss = train_step_hard(imgs, labels)        epoch_loss.update_state(loss)    print(f"Epoch {epoch+1}/{EPOCHS} - batch-hard triplet loss: {epoch_loss.result():.4f}")model.save("/content/reid_embedding_resnet50_hardmined.keras")import shutilshutil.copy("/content/reid_embedding_resnet50_hardmined.keras", "/content/drive/MyDrive/reid_embedding_resnet50_hardmined.keras")

## 7. Evaluation: Rank-1/5/10 + mAP on Market-1501 query/gallery split

In [ ]:
query_ds = Market1501Dataset(market_path, split="query")gallery_ds = Market1501Dataset(market_path, split="bounding_box_test")def compute_embeddings(dataset, model, batch_size=64, img_size=(256, 128)):    paths = [s[0] for s in dataset.samples]    person_ids = np.array([s[1] for s in dataset.samples])    camera_ids = np.array([s[2] for s in dataset.samples])    def load_batch(path_batch):        imgs = []        for p in path_batch:            img = tf.io.read_file(p)            img = tf.image.decode_jpeg(img, channels=3)            img = tf.image.resize(img, img_size)            img = preprocess_input(img)            imgs.append(img)        return tf.stack(imgs)    embeddings = []    for i in range(0, len(paths), batch_size):        batch_paths = paths[i:i+batch_size]        batch_imgs = load_batch(batch_paths)        emb = model(batch_imgs, training=False)        embeddings.append(emb.numpy())    embeddings = np.concatenate(embeddings, axis=0)    return embeddings, person_ids, camera_idsdef evaluate_reid(query_emb, query_pids, query_cids, gallery_emb, gallery_pids, gallery_cids, top_k=(1, 5, 10)):    sim_matrix = query_emb @ gallery_emb.T    num_queries = query_emb.shape[0]    rank_hits = {k: 0 for k in top_k}    avg_precisions = []    for i in range(num_queries):        sims = sim_matrix[i]        order = np.argsort(-sims)        q_pid, q_cid = query_pids[i], query_cids[i]        valid_mask = ~((gallery_pids[order] == q_pid) & (gallery_cids[order] == q_cid))        filtered_order = order[valid_mask]        filtered_pids = gallery_pids[filtered_order]        matches = (filtered_pids == q_pid).astype(int)        if matches.sum() == 0:            continue        for k in top_k:            if matches[:k].sum() > 0:                rank_hits[k] += 1        cum_matches = np.cumsum(matches)        precision_at_i = cum_matches / (np.arange(len(matches)) + 1)        ap = (precision_at_i * matches).sum() / matches.sum()        avg_precisions.append(ap)    results = {f"Rank-{k}": rank_hits[k] / num_queries for k in top_k}    results["mAP"] = np.mean(avg_precisions)    return resultsprint("Computing gallery embeddings...")gallery_emb, gallery_pids, gallery_cids = compute_embeddings(gallery_ds, model)print("Computing query embeddings...")query_emb, query_pids, query_cids = compute_embeddings(query_ds, model)results = evaluate_reid(query_emb, query_pids, query_cids, gallery_emb, gallery_pids, gallery_cids)for metric, value in results.items():    print(f"{metric}: {value:.4f}")

## 8. Face detector setup (pretrained YOLOv8n-face — no training needed)

In [ ]:
!pip install ultralytics -q

In [ ]:
import urllib.requestimport osurl = "https://huggingface.co/jalonso/yolov8face/resolve/main/yolov8n-face.pt"urllib.request.urlretrieve(url, "/content/yolov8n-face.pt")size = os.path.getsize("/content/yolov8n-face.pt")print(f"Downloaded file size: {size / 1e6:.2f} MB")

In [ ]:
from ultralytics import YOLOface_model = YOLO("/content/yolov8n-face.pt")print("Loaded:", face_model)

In [ ]:
person_model = YOLO("yolov8n.pt")print("Loaded:", person_model)

## 9. Face embedding setup (pretrained ArcFace via insightface)

In [ ]:
!pip install insightface onnxruntime-gpu -q

In [ ]:
import insightfacefrom insightface.app import FaceAnalysisface_embedder = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])face_embedder.prepare(ctx_id=0, det_size=(640, 640))print(face_embedder)

## 10. Fusion pipeline: face branch (tier 1) + re-ID fallback (tier 2)

In [ ]:
import cv2FACE_CONF_THRESHOLD = 0.5def get_reid_embedding(person_crop_bgr):    """Run the TF re-ID model on a person crop."""    img = cv2.cvtColor(person_crop_bgr, cv2.COLOR_BGR2RGB)    img = cv2.resize(img, (128, 256))    img = preprocess_input(img.astype(np.float32))    img = np.expand_dims(img, axis=0)    emb = model(img, training=False).numpy()[0]    return emb, "reid_model"def get_face_embedding(person_crop_bgr):    """Try YOLO face detection within a person crop; if confident, get ArcFace embedding."""    results = face_model.predict(person_crop_bgr, verbose=False)    boxes = results[0].boxes    if len(boxes) == 0:        return None, None    best_idx = boxes.conf.argmax().item()    conf = boxes.conf[best_idx].item()    if conf < FACE_CONF_THRESHOLD:        return None, None    x1, y1, x2, y2 = boxes.xyxy[best_idx].cpu().numpy().astype(int)    face_crop = person_crop_bgr[y1:y2, x1:x2]    faces = face_embedder.get(face_crop)    if len(faces) == 0:        return None, None    emb = faces[0].normed_embedding    return emb, "face_arcface"def identify_people(image_path):    """Basic pipeline: detect people, identify via face or re-ID, no gallery matching yet."""    img = cv2.imread(image_path)    person_results = person_model.predict(img, classes=[0], verbose=False)    person_boxes = person_results[0].boxes    identities = []    for i in range(len(person_boxes)):        x1, y1, x2, y2 = person_boxes.xyxy[i].cpu().numpy().astype(int)        person_crop = img[y1:y2, x1:x2]        emb, source = get_face_embedding(person_crop)        if emb is None:            emb, source = get_reid_embedding(person_crop)        identities.append({"bbox": (x1, y1, x2, y2), "embedding": emb, "source": source})        print(f"Person {i}: bbox=({x1},{y1},{x2},{y2}), identified via = {source}")    return identities, img

## 11. Identity gallery (named-person matching)

In [ ]:
import pickleclass IdentityGallery:    """    Stores known identities with both face and re-ID embeddings.    Matching is done within the same embedding space only    (face-to-face, re-id-to-re-id) since they're not comparable.    """    def __init__(self, face_threshold=0.4, reid_threshold=0.5):        self.face_gallery = {}        self.reid_gallery = {}        self.face_threshold = face_threshold        self.reid_threshold = reid_threshold    def enroll(self, name, person_crop_bgr):        face_emb, face_source = get_face_embedding(person_crop_bgr)        if face_emb is not None:            self.face_gallery.setdefault(name, []).append(face_emb)            print(f"Enrolled '{name}' — face embedding added")        reid_emb, _ = get_reid_embedding(person_crop_bgr)        self.reid_gallery.setdefault(name, []).append(reid_emb)        print(f"Enrolled '{name}' — re-ID embedding added")    def _cosine_sim(self, emb, gallery_embs):        gallery_matrix = np.stack(gallery_embs)        sims = gallery_matrix @ emb / (np.linalg.norm(gallery_matrix, axis=1) * np.linalg.norm(emb) + 1e-8)        return sims.max()    def match(self, emb, source):        if source == "face_arcface":            gallery, threshold = self.face_gallery, self.face_threshold        else:            gallery, threshold = self.reid_gallery, self.reid_threshold        if not gallery:            return None, 0.0        best_name, best_sim = None, -1.0        for name, embs in gallery.items():            sim = self._cosine_sim(emb, embs)            if sim > best_sim:                best_name, best_sim = name, sim        if best_sim >= threshold:            return best_name, best_sim        return None, best_sim    def save(self, path="/content/drive/MyDrive/identity_gallery.pkl"):        with open(path, "wb") as f:            pickle.dump({"face": self.face_gallery, "reid": self.reid_gallery}, f)        print(f"Gallery saved to {path}")    def load(self, path="/content/drive/MyDrive/identity_gallery.pkl"):        with open(path, "rb") as f:            data = pickle.load(f)        self.face_gallery = data["face"]        self.reid_gallery = data["reid"]        print(f"Gallery loaded: {len(self.face_gallery)} face identities, {len(self.reid_gallery)} re-id identities")gallery = IdentityGallery(face_threshold=0.4, reid_threshold=0.5)

In [ ]:
def identify_people_with_gallery(image_path, gallery):    img = cv2.imread(image_path)    person_results = person_model.predict(img, classes=[0], verbose=False)    person_boxes = person_results[0].boxes    identities = []    for i in range(len(person_boxes)):        x1, y1, x2, y2 = person_boxes.xyxy[i].cpu().numpy().astype(int)        person_crop = img[y1:y2, x1:x2]        emb, source = get_face_embedding(person_crop)        if emb is None:            emb, source = get_reid_embedding(person_crop)        name, sim = gallery.match(emb, source)        display_name = name if name else "Unknown"        identities.append({            "bbox": (x1, y1, x2, y2),            "embedding": emb,            "source": source,            "name": display_name,            "similarity": sim        })        print(f"Person {i}: {display_name} (sim={sim:.3f}, via={source})")    return identities, img

## 12. Visualization

In [ ]:
import matplotlib.pyplot as pltdef visualize_identities_named(img, identities):    vis_img = img.copy()    for i, person in enumerate(identities):        x1, y1, x2, y2 = person["bbox"]        color = (0, 255, 0) if person["name"] != "Unknown" else (0, 0, 255)        cv2.rectangle(vis_img, (x1, y1), (x2, y2), color, 2)        label = f"{person['name']} ({person['similarity']:.2f}, {person['source']})"        label_y = y1 - 10 if y1 - 10 > 10 else y2 + 20        cv2.putText(vis_img, label, (x1, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)    vis_img_rgb = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)    plt.figure(figsize=(10, 8))    plt.imshow(vis_img_rgb)    plt.axis("off")    plt.show()    return vis_img

## 13. Usage: enroll a known person, then test identification\n\nUpload photos with `from google.colab import files; files.upload()` first, then:

In [ ]:
# Example usage (update filenames to match your uploaded photos):# enroll_img = cv2.imread("/content/known_person.jpg")# gallery.enroll("Shubh", enroll_img)# identities, img = identify_people_with_gallery("/content/test_image.jpg", gallery)# vis_img = visualize_identities_named(img, identities)